In [1]:
import numpy as np
import pandas as pd

In [2]:
# Expost Attribution to Factor

factor_input = pd.read_csv("/Users/fuyuxuan/Downloads/test11_2_factor_returns.csv")
weights_input= pd.read_csv("/Users/fuyuxuan/Downloads/test11_2_weights.csv")
stock_input = pd.read_csv("/Users/fuyuxuan/Downloads/test11_2_stock_returns.csv")
beta_input= pd.read_csv("/Users/fuyuxuan/Downloads/test11_2_beta.csv")

# Convert to numpy array
F = factor_input.to_numpy(dtype=float)                # T x k
R = stock_input.to_numpy(dtype=float)                 # T x n
B = beta_input.iloc[:, 1:].to_numpy(dtype=float)      # n x k, drop Stock column
w0 = weights_input.iloc[:, 0].to_numpy(dtype=float)

# Number of periods, assets, and factors
T, n= R.shape
k_factor = F.shape[1]

# Total Return
factor_total_return = np.prod(1 + F, axis=0) - 1.0

# Compute Portfolio return series and factor/alpha contribution series
w_t = w0.copy()
factor_contri = []
alpha_contri = []
portfolio_return = []

for t in range(T):
    # Portfolio return in period t
    stock_contri_t = w_t * R[t, :]
    portfolio_return_t = np.sum(stock_contri_t)
    portfolio_return.append(portfolio_return_t)

    # Portfolio factor exposure in period t
    factor_exposure_t = w_t @ B

    # Factor contribution in period t
    factor_c_t = factor_exposure_t * F[t, :]
    factor_contri.append(factor_c_t)

    # Alpha contribution in period t
    alpha_t = portfolio_return_t - np.sum(factor_c_t)
    alpha_contri.append(alpha_t)

    # Update weights under buy-and-hold to next period
    w_t = w_t * (1 + R[t, :]) / (1 + portfolio_return_t)

factor_contri = np.array(factor_contri)
alpha_contri = np.array(alpha_contri)
portfolio_return = np.array(portfolio_return)

portfolio_total_return = np.prod(1 + portfolio_return) - 1.0
alpha_total_return = np.prod(1 + alpha_contri) - 1.0

# Return Attribution 
Rp = portfolio_total_return                            # Total portfolio return in the full horizon
k = np.log(1 + Rp) / Rp if abs(Rp) > 1e-12 else 1.0    # compute Carino linking coefficient for full-period portfolio return
k_t = np.where(np.abs(portfolio_return) > 1e-12, np.log(1 + portfolio_return) / portfolio_return, 1.0) # compute Carino coefficient for each period

factor_return_attr = np.sum(factor_contri * (k_t / k).reshape(-1, 1), axis=0)
alpha_return_attr = np.sum(alpha_contri * (k_t / k))

# Vol Attribution
vol_port = np.std(portfolio_return, ddof=1)

factor_vol_attr = np.array([
    np.cov(factor_contri[:, j], portfolio_return, ddof=1)[0, 1] / vol_port
    for j in range(k_factor)
])

alpha_vol_attr = np.cov(alpha_contri, portfolio_return, ddof=1)[0, 1] / vol_port

output = pd.DataFrame({
    "Value": ["TotalReturn", "Return Attribution", "Vol Attribution"],
    "F1": [factor_total_return[0], factor_return_attr[0], factor_vol_attr[0]],
    "F2": [factor_total_return[1], factor_return_attr[1], factor_vol_attr[1]],
    "F3": [factor_total_return[2], factor_return_attr[2], factor_vol_attr[2]],
    "Alpha": [alpha_total_return, alpha_return_attr, alpha_vol_attr],
    "Portfolio": [portfolio_total_return, np.sum(factor_return_attr) + alpha_return_attr, vol_port]
})

print(output)

                Value        F1        F2        F3     Alpha  Portfolio
0         TotalReturn  0.079939  0.026711 -0.016317  0.046443   0.105327
1  Return Attribution  0.065173  0.000174 -0.007579  0.047559   0.105327
2     Vol Attribution  0.005013 -0.000009  0.001785  0.005445   0.012234
